# Delegation, Token Exchange, and Confused-Deputy Prevention

**Goal:** exchange a parent access token for a signed, shorter-lived, audience-bound child, then make the downstream API independently enforce caller, actor, transaction, and live resource policy.

This notebook uses ephemeral keys and no external credentials. It is a learning profile, not a production authorization server.

## 1. Load the executable lab

The notebook imports the same implementation exercised by the focused test suite. Fixed time keeps every result reproducible.

In [ ]:
import asyncio
import runpy
from datetime import UTC, datetime
from pathlib import Path

lab = runpy.run_path(Path("lab.py"))
NOW = datetime(2026, 9, 27, 17, 0, tzinfo=UTC)
sts, ticket_api, parent_token, support_actor, ticket_actor = lab["default_environment"](NOW)
print("Loaded signed, credential-free delegation environment")

## 2. Security thesis and trust boundaries

Effective child authority is the intersection of the request, parent grant, authenticated actor policy, target policy, and the downstream API's current record policy. RFC 8693 defines the exchange protocol; this local profile implements narrowing, transaction binding, depth, and lineage rules.

The workload identity below represents prior mTLS or private-key authentication. It is trusted application context, not a tool argument.

In [ ]:
parent_claims = sts.decode_for_resource(parent_token, audience=lab["MCP_RESOURCE"], now=NOW)
safe_parent = {key: parent_claims[key] for key in (
    "iss", "sub", "aud", "scope", "tenant_id", "purpose",
    "resource_ids", "delegation_depth", "max_delegation_depth", "jti"
)}
{"authenticated_actor": support_actor.model_dump(), "parent": safe_parent}

## 3. Keep authority out of the MCP tool schema

The model may select a ticket ID. It may not choose the token, subject, tenant, workload actor, audience, scopes, purpose, or transaction identity.

In [ ]:
service = lab["DelegatingSupportService"](sts, ticket_api, parent_token, support_actor, NOW)
server = lab["build_mcp_server"](service)

async def get_tool_schema():
    async with lab["Client"](server) as client:
        tools = await client.list_tools()
        return tools.tools[0].input_schema

tool_schema = await get_tool_schema()
tool_schema

## 4. Exchange for one downstream resource

The request is RFC 8693-shaped and adds local resource, purpose, transaction, lifetime, and idempotency constraints. The response contains a bearer access token but no refresh token.

In [ ]:
request = lab["exchange_request"](
    parent_token,
    exchange_id="exchange-notebook-1",
    scopes=frozenset({"ticket:read"}),
    resource_ids=("acme-100",),
    transaction_id="txn-notebook-1",
    ttl=60,
)
response = sts.exchange(request, actor=support_actor, now=NOW)
child_token = response.access_token.get_secret_value()
child_claims = sts.decode_for_resource(child_token, audience=lab["TICKETS_API"], now=NOW)
{
    "response": {"issued_token_type": response.issued_token_type, "token_type": response.token_type, "expires_in": response.expires_in, "scope": response.scope},
    "child": {key: child_claims[key] for key in ("sub", "aud", "scope", "resource_ids", "purpose", "transaction_id", "delegation_depth", "parent_jti", "act")},
}

### Interpret the child

- The original analyst remains the subject.
- The support MCP workload is the current actor.
- The audience changes from the MCP resource to the ticket API.
- Scope and resource IDs are narrower, lifetime is 60 seconds, and depth increases to one.
- The raw token is never printed.

## 5. Enforce independently at the ticket API

The API verifies signature, issuer, exact audience, time, presenter-to-actor binding, lineage, transaction, purpose, scope, resource ID, depth, tenant, and current ticket assignment.

In [ ]:
allowed = ticket_api.read_ticket(
    token=child_token,
    presenter=support_actor,
    ticket_id="acme-100",
    transaction_id="txn-notebook-1",
    now=NOW,
)
allowed

## 6. Attack matrix: attempts are not violations

Each case asks the STS to expand one dimension. A denied attempt is useful telemetry, but it is not an unauthorized downstream effect. The target unexpected-allow rate is zero.

In [ ]:
negative_cases = [
    ("scope expansion", {"scopes": frozenset({"ticket:write"})}),
    ("resource expansion", {"resource_ids": ("globex-200",)}),
    ("purpose drift", {"purpose": "analytics"}),
    ("target drift", {"resource": "https://admin.example/api"}),
    ("lifetime expansion", {"ttl": 100}),
]
results = []
for index, (name, updates) in enumerate(negative_cases, start=1):
    try:
        attack = lab["exchange_request"](parent_token, exchange_id=f"exchange-attack-{index}", **updates)
        sts.exchange(attack, actor=support_actor, now=NOW)
        results.append({"case": name, "outcome": "UNEXPECTED_ALLOW"})
    except lab["ExchangeDenied"] as exc:
        results.append({"case": name, "outcome": "DENY", "reason": exc.code})

unexpected_allows = sum(item["outcome"] == "UNEXPECTED_ALLOW" for item in results)
{"results": results, "unexpected_allow_rate": unexpected_allows / len(results)}

## 7. Prove token passthrough fails

The incoming parent token is valid for the MCP resource, not the ticket API. Forwarding it is both an audience error and prohibited MCP token passthrough.

In [ ]:
try:
    ticket_api.read_ticket(
        token=parent_token,
        presenter=support_actor,
        ticket_id="acme-100",
        transaction_id="txn-passthrough",
        now=NOW,
    )
    passthrough = "UNEXPECTED_ALLOW"
except lab["DownstreamDenied"] as exc:
    passthrough = exc.code
passthrough

## 8. Prove the bearer-token limitation

A token ID, short expiry, and transaction binding do not create proof of possession. The same valid child bearer is accepted twice. Production can add DPoP or mTLS sender constraints where risk warrants it.

In [ ]:
replay_one = ticket_api.read_ticket(token=child_token, presenter=support_actor, ticket_id="acme-100", transaction_id="txn-notebook-1", now=NOW)
replay_two = ticket_api.read_ticket(token=child_token, presenter=support_actor, ticket_id="acme-100", transaction_id="txn-notebook-1", now=NOW)
{"same_result": replay_one == replay_two, "accepted_fingerprints": ticket_api.accepted_token_fingerprints[-2:]}

## 9. Build a two-hop actor chain

The ticket workload exchanges the first child for an even narrower audit token. The outer actor is current; the nested actor is history. Transaction identity cannot drift between hops.

In [ ]:
chain_first_request = lab["exchange_request"](
    parent_token,
    exchange_id="exchange-chain-1",
    scopes=frozenset({"ticket:read", "trace:append"}),
    transaction_id="txn-chain-1",
)
chain_first = sts.exchange(chain_first_request, actor=support_actor, now=NOW)
chain_second_request = lab["exchange_request"](
    chain_first.access_token.get_secret_value(),
    exchange_id="exchange-chain-2",
    resource=lab["AUDIT_API"],
    scopes=frozenset({"trace:append"}),
    transaction_id="txn-chain-1",
    ttl=30,
)
chain_second = sts.exchange(chain_second_request, actor=ticket_actor, now=NOW)
chain_claims = sts.decode_for_resource(chain_second.access_token.get_secret_value(), audience=lab["AUDIT_API"], now=NOW)
{"scope": chain_claims["scope"], "depth": chain_claims["delegation_depth"], "actor_chain": chain_claims["act"]}

## 10. Revoke the root and retest the child

This fixture defines explicit recursive lineage revocation. That behavior comes from the online status policy below—not automatically from JWT or RFC 8693.

In [ ]:
sts.revoke("root-grant-1")
try:
    ticket_api.read_ticket(token=child_token, presenter=support_actor, ticket_id="acme-100", transaction_id="txn-notebook-1", now=NOW)
    revocation_result = "UNEXPECTED_ALLOW"
except lab["DownstreamDenied"] as exc:
    revocation_result = exc.code
revocation_result

## 11. Run the complete real-MCP scenario

This final run uses an isolated environment and the official SDK's in-memory client to call the actual MCP tool, test a cross-tenant request, and return redacted evidence.

In [ ]:
evidence = await lab["run_scenario"]()
evidence

## Production upgrades

- Replace the in-process STS with a maintained identity platform and exact exchange policy.
- Authenticate workloads with mTLS, private-key JWT, or workload identity.
- Add safe key rotation, durable idempotency, rate limits, protected audit storage, and tested outage behavior.
- Define revocation latency and descendant semantics explicitly.
- Add DPoP or mTLS-bound access tokens if bearer replay exceeds the risk budget.

## Reflection

1. Which guarantees in this notebook come from RFC 8693, and which come from the local profile?
2. Why must the downstream API check the current ticket assignment after validating the child?
3. What would have to change to make the observed replay fail?
4. In the nested `act` claim, which actor is current and which is only historical?